# GI Lab: Weisfeiler–Leman experiments

In Colab, choose **Runtime → Run all** from a fresh runtime. The first code cell downloads the repository, installs plotting support, and imports `Graph` and the algorithms before the examples run. Do not start by running an example cell on its own.

This workbench compares 1-WL and 2-WL graph-isomorphism tests, explores examples, and checks small graphs with an exact oracle.

In [ ]:
# Find a local checkout or download the public repository (Colab starts without the package installed).
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "gi_lab").is_dir():
    ROOT = ROOT / "gi-lab-repository"
    if not (ROOT / "gi_lab").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/shrnasa011/testing_isomorphism.git", str(ROOT)],
            check=True,
        )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%pip install -q matplotlib

from gi_lab.algorithms import ColorRefinement, TwoWL
from gi_lab.exact import is_isomorphic
from gi_lab.families import (
    cycle, disjoint_cycles, rook_graph, shrikhande_graph,
    default_corpus,
)
from gi_lab.experiments import run
from gi_lab.graph import Graph
from gi_lab.search import find_collision
from gi_lab.draw import draw_pair


## Build graphs by hand

`Graph.from_edges` is the most direct way to encode a small example. Isolated vertices can be supplied explicitly.

In [ ]:
triangle_plus_isolate = Graph.from_edges(
    [(0, 1), (1, 2), (2, 0)],
    vertices=range(4),
)
triangle_plus_isolate.describe(), triangle_plus_isolate.edge_list()

## Compare a known 1-WL failure

The 12-cycle and two disjoint 6-cycles are not isomorphic, but every vertex has degree two. Therefore 1-WL reaches the same single-colour partition. 2-WL distinguishes them.

In [ ]:
left, right = cycle(12), disjoint_cycles(6, 6)
for algorithm in (ColorRefinement(), TwoWL()):
    same = algorithm.refine(left).signature == algorithm.refine(right).signature
    print(f'{algorithm.name}: same signature = {same}')
print('exactly isomorphic:', is_isomorphic(left, right))
draw_pair(left, right, ('Random graph A', 'Random graph B'))

## Run the curated regression corpus

A `collision` means a non-isomorphic pair that this invariant cannot separate; it is not a bug in the runner.

In [ ]:
trials = run(default_corpus(include_permutations=True), [ColorRefinement(), TwoWL()])
for trial in trials:
    print(f'{trial.case:26} {trial.algorithm:3}  {trial.verdict}')

## Search randomly for a test-breaking pair

Random regular graphs are a productive source of 1-WL collisions because all vertices begin with the same degree. The seed makes the search repeatable. The exact oracle validates any reported collision, and the edge lists let you turn it into a permanent hand-built case.

Pass `connected_only=True` to `find_collision` to skip disconnected samples and search for collisions among connected graphs only.

For Erdős–Rényi samples, use `model="erdos"` and either `average_degree=d` (which sets `p = d / (vertices - 1)`) or `probability=p`. These options are mutually exclusive.

In [ ]:
collision = find_collision(
    ColorRefinement(), vertices=10, degree=2, samples=40, seed=7
)
if collision is None:
    print('No collision found within this budget; try more samples or a new seed.')
else:
    print(collision.reproduce())
    draw_pair(collision.left, collision.right, ('Random graph A', 'Random graph B'))

### Search Erdős–Rényi graphs by expected average degree

Here `average_degree=4` sets the edge probability to `4 / (10 - 1)`. `connected_only=True` skips disconnected samples.

In [ ]:
erdos_collision = find_collision(
    ColorRefinement(), vertices=5, model="erdos",
    average_degree=3, samples=400, seed=7, connected_only=True
)
if erdos_collision is None:
    print("No collision found within this sample budget.")
else:
    print(erdos_collision.reproduce())
    draw_pair(erdos_collision.left, erdos_collision.right,
              ("Erdos graph A", "Erdos graph B"))

## A stronger regular-graph example

The rook and Shrikhande graphs are non-isomorphic strongly regular graphs with the same parameters. They collide under both included WL variants.

In [ ]:
rook, shrikhande = rook_graph(), shrikhande_graph()
print('exactly isomorphic:', is_isomorphic(rook, shrikhande))
for algorithm in (ColorRefinement(), TwoWL()):
    print(algorithm.name, algorithm.refine(rook).signature == algorithm.refine(shrikhande).signature)
draw_pair(rook, shrikhande, ('4×4 rook graph', 'Shrikhande graph'))